## Shared Config (if needed)

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from ravelry_scraper.constants import DATA_DIR


In [ ]:
con = duckdb.connect()

parquet_files = list(DATA_DIR.glob("**/*.parquet"))

# Register Parquet datasets as SQL views
for file in parquet_files:
    name = f"{file.parent.name}_{file.stem}"
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{file}')
    """)


In [ ]:
# Store list of all DuckDB views
duckdb_views = [row[0] for row in con.execute("""
    SELECT view_name
    FROM duckdb_views
""").fetchall()]

# Print all DuckDB views to verify the list
print(duckdb_views)

In [ ]:
pl.Config.set_fmt_float("full")
pl.Config.set_tbl_rows(25)
pl.Config.set_thousands_separator(",")

# Projects

Ravelry's API exposes an endpoint for `GET /projects/search.json` that returns user-created Project records. Each Project can be linked to one Pattern through `pattern_id`. This will let us combine pattern characteristics with project-level outcomes like completion status, time to finish, and overall engagement. Below, you can see all columns for both the raw Project data and the cleaned Project data. Cleaning focused on dropping unused columns, standardizing data types, and extracting nested records into separate tables. From the cleaned data, notable columns include:

| **Notable Fields** | **Use Cases** |
| --- | --- |
| `id`, `pattern_id`, `pattern_name` | Identify Projects and join them to Patterns |
| `project_status_id`, `status_name`, `progress` | Measure project outcomes, like % marked completed |
| `started`, `completed` | Estimate time to completion |
| `comments_count`, `favorites_count`, `photos_count` | Analyze user engagement |

### Documentation
- https://www.ravelry.com/api#projects_search
- https://www.ravelry.com/api#Project__result


## Data Cleaning

First, we can confirm that cleaning the data did not remove any records:

In [ ]:
display(
    con.sql("""
        SELECT
            (SELECT COUNT(*) FROM raw_projects) AS raw_count,
            (SELECT COUNT(*) FROM clean_projects) AS clean_count,
            (SELECT (SELECT COUNT(*) FROM raw_projects) - (SELECT COUNT(*) FROM clean_projects)) AS difference
    """).pl()
)

Second, we can compare the two schemas to see which columns were dropped and how data types were standardized:

In [ ]:
display(
    con.sql("""
        WITH raw_schema AS (
            DESCRIBE raw_projects
        ),
        clean_schema AS (
            DESCRIBE clean_projects
        )
        SELECT 
            raw_schema.column_name AS column_name,
            CASE
                WHEN raw_schema.column_type LIKE 'STRUCT%' THEN 'STRUCT()'
                ELSE raw_schema.column_type
            END AS raw_type,
            CASE
                WHEN raw_schema.column_name = clean_schema.column_name THEN clean_schema.column_type
                ELSE '<DROPPED>'
            END AS clean_type
        FROM raw_schema
        LEFT JOIN clean_schema
        ON raw_schema.column_name = clean_schema.column_name
    """).df()
)


By viewing a few lines, we can get a better sense for the dataset we weill be working with:

In [ ]:
display(
    con.sql("""
        SELECT * 
        FROM clean_projects
        LIMIT 10
    """).pl()
)

## Data Exploration

TODO - describe

### Summary Statistics

TODO - Describe

In [ ]:
display(
    con.sql("""
        SELECT
            CAST(SUM(clean_patterns.projects_count) AS BIGINT) AS total_projects,
            COUNT(clean_projects.*) AS scraped_projects,
            COUNT(clean_projects.pattern_id) AS with_pattern,
            COUNT(DISTINCT clean_projects.pattern_id) AS unique_patterns,
            COUNT(clean_projects.completed) AS completed,
            ROUND(
                AVG(
                    CAST(clean_projects.completed AS DATE) - CAST(clean_projects.started AS DATE)
                )
            , 2) AS avg_completion_days
        FROM clean_projects
        JOIN clean_patterns
        ON clean_projects.pattern_id = clean_patterns.id
    """).pl()
)

### Most Popular Patterns

TODO - Describe

In [ ]:
# Most Popular Patterns (using Projects dataset)
most_popular_patterns_df = con.sql("""
    SELECT
        p.pattern_id,
        r.name AS pattern_name,
        COUNT(p.*) AS num_projects,
        RANK() OVER (ORDER BY num_projects DESC) AS rank,
    FROM clean_projects AS p
    JOIN raw_patterns AS r
        ON p.pattern_id = r.id
    GROUP BY p.pattern_id, r.name
    ORDER BY num_projects DESC
    LIMIT 10
""").pl()

display(most_popular_patterns_df)

TODO - describe

In [ ]:
# Most Popular Patterns (using Patterns dataset)
most_popular_patterns_df_2 = con.sql("""
    SELECT
        id,
        name AS pattern_name,
        projects_count AS num_projects,
        RANK() OVER (ORDER BY num_projects DESC) AS rank,
    FROM raw_patterns
    ORDER BY num_projects DESC
    LIMIT 10
""").pl()

pl.concat(
    [
        most_popular_patterns_df.select(
            pl.col('rank').alias('rank'),
            pl.col('pattern_name').alias('project_pattern_name'),
            pl.col('num_projects').alias('dataset_count'),
        ),
        most_popular_patterns_df_2.drop('id').select(
            pl.col('pattern_name').alias('pattern_name'),
            pl.col('num_projects').alias('projects_count_col'),
        ),
    ],
    how='horizontal'
)

### Most Popular by Year

TODO - describe

In [ ]:
# Most Popular Project Pattern by Year
most_popular_pattern_by_year_df = con.sql("""
    WITH yearly_counts AS (
        SELECT
            pattern_id,
            COUNT(*) AS num_projects,
            EXTRACT(YEAR FROM created_at) AS year
        FROM clean_projects
        GROUP BY year, pattern_id
        ORDER BY year DESC
    ),
    ranked_counts AS (
        SELECT
            pattern_id,
            num_projects,
            year,
            RANK() OVER (PARTITION BY year ORDER BY num_projects DESC) AS rank
        FROM yearly_counts
    )
    SELECT 
        ranks.year,
        patterns.name AS most_popular_pattern,
        ranks.num_projects,
    FROM ranked_counts AS ranks
    JOIN raw_patterns AS patterns
        ON ranks.pattern_id = patterns.id
    WHERE rank = 1
    ORDER BY year DESC
""").df()

display(most_popular_pattern_by_year_df.set_index("year"))

In [ ]:
counts_df = most_popular_pattern_by_year_df["most_popular_pattern"].value_counts().reset_index()

plt.figure(figsize=(8, 6))
ax = sns.barplot(
    orient="h",
    data=counts_df,
    x="count",
    y="most_popular_pattern",
    hue="most_popular_pattern",
    palette="deep",
)

ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_xlabel("Number of Years as Most Popular Pattern")
ax.set_ylabel("")
ax.set_title("Times as Pattern of the Year")

sns.despine()
plt.tight_layout()
plt.show()

### Average Completion Time by Year

TODO - describe

In [ ]:
# Average Completion Time Variation by Year
avg_completion_time_df = con.sql("""
    WITH yearly_stats AS (
        SELECT
            EXTRACT(YEAR FROM created_at) AS year,
            ROUND(
                AVG(
                    CAST(completed AS DATE) - CAST(started AS DATE)
                )
                , 2
            ) AS avg_completion_time
        FROM clean_projects
        WHERE (
            completed IS NOT NULL AND 
            started IS NOT NULL AND
            completed >= started
        )
        GROUP BY year
        ORDER BY year
    )
    SELECT
        year,
        avg_completion_time,
        ROUND(
            avg_completion_time - LAG(avg_completion_time) OVER (ORDER BY year ASC), 
            2
        ) AS change_in_days,
        ROUND(
            ((avg_completion_time - LAG(avg_completion_time) OVER (ORDER BY year ASC)) 
            / NULLIF(LAG(avg_completion_time) OVER (ORDER BY year ASC), 0)) * 100, 
            2
        ) AS change_in_percentage
    FROM yearly_stats
    ORDER BY year
""").df()

display(avg_completion_time_df)

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.lineplot(
    data=avg_completion_time_df,
    x="year",
    y="avg_completion_time",
)

ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_xlabel("Year")
ax.set_ylabel("Average Number of Days to Complete")
ax.set_title("Average Completion Time by Year")

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.lineplot(
    data=avg_completion_time_df,
    x="year",
    y="change_in_days",
)

ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Year")
ax.set_ylabel("Change in Days")
ax.set_title("Change in Average Completion Time by Year")

sns.despine()
plt.tight_layout()
plt.show()

### Correlation Matrices

TODO - Discussion


Notes:

Raw Numbers vs. Log-Transformed Numbers


Aspect	Raw Numbers (favorites_count)	Log Numbers (ln(1.0 + favorites_count))
Distribution Shape	Heavily right-skewed (long tail of viral hits).	Compressed, closer to symmetric / normal.
Sensitivity to Outliers	Extreme. A single viral pattern with 20,000 favorites and 1,000 comments can artificially inflate the correlation from 0.3 to 0.85.	Low. Extreme values are pulled inward, preventing a handful of outliers from dominating the math.
Relationship Captured	Additive / Absolute: Does +100 favorites equal +10 comments across all levels?	Multiplicative / Relative: Does a 10% jump in favorites correspond to a proportional jump in comments?
Behavior Near Zero	Linear spacing (0, 1, 2, 3...).	Spreads out small counts (0, 1, 2, 5), making differences among typical projects visible.

Why rating Behaves Differently Here

Notice that in your SQL query, rating is bounded (typically on a 1–5 scale) rather than an unbounded engagement count:
• Bounded scales aren't right-skewed: Applying ln(1 + rating) to a 1–5 rating compresses the top end (4.0 vs. 5.0 becomes 1.61 vs. 1.79), which rarely improves linearity and makes the correlation harder to interpret.
• Best practice: Keep rating in its raw form and compare it against both raw counts and log counts (ln(1 + favorites_count)). In fact, comparing corr(rating, favorites) vs. corr(rating, log_favorites) often reveals that raw correlation is near zero (because viral patterns attract casual raters that pull averages toward 4.5), whereas log correlation captures the baseline relationship across typical projects.

Summary Rule of Thumb

• Use Raw Numbers when you care about total volume and linear scale (e.g., predicting total comment load from total favorites) or when a variable is already bounded (like rating).
• Use Log Numbers (ln(1 + x)) when analyzing skewed count data to uncover underlying proportional relationships without letting viral outliers skew the entire matrix.
• Alternative Check (Spearman Rank Correlation): Because ln(1 + x) is a strictly increasing (monotonic) transformation, running a Spearman correlation (df.corr(method="spearman")) on the raw numbers will give you a similar outlier-resistant view without needing log columns at all.

In [ ]:
df = con.sql("""
    SELECT
        favorites_count,
        comments_count,
        photos_count,
        rating
    FROM clean_projects
""").df()

corr_matrix_1 = df.corr()
corr_matrix_2 = df.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
heatmap_settings = {
    "annot": True,
    "fmt": ".2f",
    "cmap": sns.blend_palette([
        "#e7f0f7",
        "#5a0e21"
    ], as_cmap=True),
    "square": True,
    "vmax": 1,
    "vmin": 0,
}

sns.heatmap(corr_matrix_1, ax=axes[0], **heatmap_settings)
axes[0].set_title("Raw Count (Pearson) Correlation")

sns.heatmap(corr_matrix_2, ax=axes[1], **heatmap_settings)
axes[1].set_title("Log-transformed (Spearman) Correlation")

plt.tight_layout()
plt.show()


## Challenges and Limitations

Unlike other endpoints, Projects cannot be accessed sequentially via ID. To overcome this obstacle, the `search_pipeline` was run with various queries and parameters. For example, we identified all `sort` options and ran in both forward and reverse order via `run_search_pipeline_for_all_sort_orders`. We also identified the top ~25 Pattern names and queried for these in `run_search_pipeline_for_all_queries`. Because these queries typically returned less than the maximum number of pages (2084), we do not need to run this search in reverse order.

Unfortunately, we still cannot know the exact proportion of sampling. Ravelry's search page shows there are over 33 million Projects.[1] Summing the `projects_count` column in the Pattern dataset, we calculate there are ### million Projects. This would mean our dataset is a sample of (a maximum of) ### % of total Projects.

[1] Ravelry. (n.d.).  Projects Search. Retrieved November 15, 2026 from https://www.ravelry.com/projects/search
